# Лекция 12. HTTP и асинхронное программирование

Когда программа получает курсы валют, данные репозитория или ответ внутреннего сервиса, большая часть времени уходит не на вычисления, а на ожидание сети. Сначала разберём, чем именно обмениваются HTTP-клиент и сервер, затем научимся перекрывать независимые ожидания через `asyncio`. Асинхронность здесь не фокус и не способ ускорить любую функцию: это организация времени ожидания.

## Цели

После лекции вы сможете:

- читать HTTP-запрос и ответ: метод, URL, заголовки, статус и тело;
- отличать HTTP от JSON и сетевую ошибку от ошибочного статуса;
- отправлять запрос с тайм-аутом и явной проверкой результата;
- различать CPU-bound и I/O-bound работу;
- объяснять coroutine, task, event loop и точку переключения `await`;
- запускать связанные задачи через `asyncio.TaskGroup`;
- повторно использовать `aiohttp.ClientSession`;
- ограничивать время и конкурентность операций;
- корректно освобождать ресурсы при отмене;
- замечать блокирующий код внутри асинхронной программы.

## Перед началом

Нужны функции, исключения, контекстные менеджеры и JSON из прошлых занятий. Целевая версия — Python 3.14. На HTTP и JSON заложено около 25 минут, на виды нагрузки — 15 минут, на coroutine/task/event loop — 20 минут, на `TaskGroup` и реальный HTTP-клиент — 15 минут, на тайм-аут, отмену, semaphore, необычные случаи и вопросы — 15 минут.

Сетевые ячейки помечены тегом `network`: их результат зависит от подключения, доступности GitHub API и его лимита запросов.

## HTTP: запрос и ответ

HTTP — протокол клиент-серверного обмена. Клиент инициирует **запрос**, сервер возвращает **ответ**. У запроса есть:

- метод, например `GET` или `POST`;
- URL ресурса;
- заголовки с метаданными;
- иногда тело.

У ответа есть числовой статус, заголовки и, возможно, тело. Соединение, DNS и TLS важны для доставки, но прикладной код чаще работает уже с этой моделью сообщения.

## Из чего состоит URL

В URL `https://api.example.test:443/repos/python/cpython?page=2#readme`:

- `https` — схема;
- `api.example.test` — host;
- `443` — порт;
- `/repos/python/cpython` — путь;
- `page=2` — query-параметр;
- `readme` — fragment, который обычно не отправляется HTTP-серверу.

Параметры нельзя надёжно собирать конкатенацией: пробелы, кириллица, `&` и повторяющиеся значения требуют кодирования.

In [ ]:
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

url = "https://api.example.test/search?lang=ru#results"
parts = urlsplit(url)
query = parse_qsl(parts.query, keep_blank_values=True)
query.extend([("q", "курс Python"), ("tag", "http"), ("tag", "async")])
new_url = urlunsplit(parts._replace(query=urlencode(query)))
assert new_url.endswith("q=%D0%BA%D1%83%D1%80%D1%81+Python&tag=http&tag=async#results")

## Методы выражают намерение

- `GET` получает представление ресурса и не должен изменять его состояние.
- `POST` отправляет данные для обработки или создаёт подчинённый ресурс.
- `PUT` обычно заменяет ресурс по известному адресу.
- `PATCH` частично изменяет ресурс.
- `DELETE` просит удалить ресурс.

Это семантический договор, а не защита на уровне синтаксиса. Сервер технически может удалить запись по `GET`, но такой API ломает ожидания клиентов, кэширования и повторов. Идемпотентность означает, что повтор одинакового запроса имеет тот же целевой эффект; это не означает буквально одинаковое тело ответа.

## Статус — часть результата

Первая цифра задаёт класс:

- `2xx` — запрос обработан успешно;
- `3xx` — перенаправление или использование кэша;
- `4xx` — запрос клиента нельзя выполнить в текущем виде;
- `5xx` — сервер не смог корректно обработать запрос.

Полезно различать `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`, `409 Conflict`, `429 Too Many Requests`, `500 Internal Server Error` и `503 Service Unavailable`. Ответ `404` дошёл по сети успешно: это HTTP-ошибка, а не отсутствие ответа.

## Заголовки, тело и JSON

Заголовок `Content-Type` описывает формат тела ответа, `Accept` — предпочитаемый клиентом формат, `Authorization` несёт учётные данные, а `Retry-After` может подсказать время повтора. Имена HTTP-заголовков регистронезависимы.

JSON — формат данных, а не сетевой протокол. JSON-объект превращается в `dict`, массив — в `list`, `null` — в `None`. Синтаксически корректный JSON ещё не гарантирует нужную схему: поле может отсутствовать или иметь неожиданный тип. Поэтому существуют два разных шага — разобрать JSON и проверить предметные данные.

In [ ]:
import json

text = '{"full_name": "python/cpython", "language": "Python", "archived": false}'
payload = json.loads(text)
assert payload["full_name"] == "python/cpython"
assert payload["archived"] is False

## Простой синхронный HTTP-клиент

Для одного запроса библиотека `requests` даёт прямую модель. Всегда задаём тайм-аут: чужой сервис может не ответить. Затем явно проверяем статус через `raise_for_status()` и только после этого разбираем JSON.

Тайм-аут, HTTP-ошибка и ошибка декодирования — разные сбои. Ловить общий `Exception` и возвращать пустой словарь опасно: вызывающий код не отличит пустой результат от поломки инфраструктуры.

In [ ]:
import requests

response = requests.get(
    "https://api.github.com/repos/python/cpython",
    headers={"Accept": "application/vnd.github+json", "User-Agent": "hse-python-course"},
    timeout=10,
)
response.raise_for_status()
repo = response.json()
summary = {
    "full_name": repo["full_name"],
    "stars": repo["stargazers_count"],
    "language": repo["language"],
}
assert summary["full_name"] == "python/cpython"

Для серии запросов нужен клиент или session, который повторно использует соединения и общие заголовки. Нельзя создавать новую сессию на каждый URL.

Автоматический повтор тоже не универсален. Повтор `GET` после временного `503` обычно разумен с ограничением числа попыток и паузой. Слепой повтор `POST` может создать две оплаты или две заявки, если сервер уже выполнил первый запрос, но ответ потерялся. Политика повторов следует из семантики операции.

## CPU-bound и I/O-bound работа

**CPU-bound** задача большую часть времени вычисляет: сортирует большой массив, кодирует видео, перебирает варианты. **I/O-bound** задача в основном ждёт: сеть, диск, базу данных, очередь сообщений.

`asyncio` особенно полезен для большого числа независимых ожиданий. Пока одна операция ждёт сокет, event loop запускает другую готовую задачу. Но если coroutine на секунду занялась чистым Python-вычислением без `await`, на эту секунду остановятся остальные задачи того же event loop. Для CPU-bound работы обычно выбирают процессы или библиотеку, выполняющую вычисление вне интерпретатора.

## Последовательно, конкурентно, параллельно

- **Последовательно:** вторая работа начинается после полного завершения первой.
- **Конкурентно:** несколько работ находятся в процессе и чередуются.
- **Параллельно:** несколько работ физически выполняются одновременно на разных вычислительных ресурсах.

Один event loop обычно даёт конкурентность в одном потоке. Для сетевого ожидания этого достаточно: процессор всё равно не вычисляет ответ удалённого сервера. Асинхронность экономит время только там, где независимые ожидания можно перекрыть.

## Coroutine и `await`

`async def` объявляет coroutine-функцию. Её вызов не выполняет тело, а создаёт coroutine-объект. Чтобы код начал работать, объект нужно `await`-ить или превратить в task.

`await` не означает «обязательно переключиться». Он передаёт управление event loop, если ожидаемая операция ещё не готова. Переключение кооперативное: сама coroutine доходит до точки ожидания и разрешает другим продолжить.

In [ ]:
import asyncio

async def load_report(name: str, delay: float) -> str:
    await asyncio.sleep(delay)  # управляемая модель ожидания I/O
    return f"{name}: ready"

coroutine_object = load_report("sales", 0)
assert not isinstance(coroutine_object, asyncio.Task)
assert await coroutine_object == "sales: ready"

## Event loop и граница программы

Event loop хранит готовые задачи, ждёт событий I/O и возобновляет coroutine после завершения ожидаемой операции. В обычном скрипте верхнеуровневая граница выглядит как `asyncio.run(main())`. В Jupyter event loop уже работает, поэтому в ячейке доступен top-level `await main()`.

В прикладном коде редко нужно вручную создавать loop. Внутри coroutine его получают через `asyncio.get_running_loop()`.

> **Изменилось в Python 3.14.** `asyncio.get_event_loop()` теперь поднимает `RuntimeError`, если текущий loop не установлен. Для нового кода правило проще: `asyncio.run()` на синхронной границе, `get_running_loop()` — внутри async-кода. Система политик event loop устарела и должна быть удалена в Python 3.16.

## Task запускает coroutine конкурентно

Task связывает coroutine с event loop и хранит её будущее состояние: результат, исключение или отмену. `asyncio.create_task(coro())` планирует выполнение и сразу возвращает объект задачи. Сильную ссылку на task нужно сохранить и затем дождаться. Создать task и забыть о ней — значит потерять место обработки ошибки и момент завершения.

Если просто написать два `await` подряд, второй вызов не начнётся, пока не закончится первый.

In [ ]:
from time import perf_counter

async def sequential_reports() -> list[str]:
    return [
        await load_report("sales", 0.04),
        await load_report("costs", 0.03),
    ]

async def concurrent_reports() -> list[str]:
    first = asyncio.create_task(load_report("sales", 0.04))
    second = asyncio.create_task(load_report("costs", 0.03))
    return [await first, await second]

start = perf_counter()
assert await sequential_reports() == await concurrent_reports()
# Конкурентный вариант близок к максимуму задержек, последовательный — к сумме.

## `TaskGroup`: связанные задачи живут вместе

> **Появилось в Python 3.11.** `asyncio.TaskGroup` даёт структурированную конкурентность. Все созданные в группе задачи дожидаются при выходе из `async with`. Если одна завершается обычным исключением, группа отменяет оставшиеся, ждёт их очистку и поднимает собранную ошибку.

Это безопаснее набора забытых `create_task`: время жизни дочерних задач ограничено видимым блоком. Результаты можно прочитать из сохранённых объектов task после выхода.

In [ ]:
async def reports_with_group(names: list[str]) -> list[str]:
    tasks = []
    async with asyncio.TaskGroup() as group:
        for index, name in enumerate(names):
            tasks.append(group.create_task(load_report(name, 0.01 * index)))
    return [task.result() for task in tasks]

assert await reports_with_group(["sales", "costs", "profit"]) == [
    "sales: ready", "costs: ready", "profit: ready"
]

> **Изменилось в Python 3.14.** `TaskGroup.create_task()` передаёт все дополнительные именованные аргументы в `loop.create_task()`. Для базового использования ничего менять не нужно; важнее помнить старую отметку: сам `TaskGroup` появился в 3.11.

## Реальные конкурентные HTTP-запросы

`requests` — синхронная библиотека: её вызов блокирует поток до ответа. Для async-программы используем асинхронный клиент, например `aiohttp`. Одна `ClientSession` владеет пулом соединений и живёт дольше отдельного запроса.

У `aiohttp` отдельно ожидаются получение ответа и чтение тела. Оба ресурса оформляются через `async with`, поэтому соединение корректно возвращается в пул даже при ошибке.

In [ ]:
import aiohttp

async def fetch_repo(session: aiohttp.ClientSession, full_name: str) -> dict:
    url = f"https://api.github.com/repos/{full_name}"
    async with session.get(url) as response:
        response.raise_for_status()
        payload = await response.json()
        return {
            "full_name": payload["full_name"],
            "stars": payload["stargazers_count"],
        }

async def fetch_repositories(names: list[str]) -> list[dict]:
    timeout = aiohttp.ClientTimeout(total=10)
    headers = {"Accept": "application/vnd.github+json", "User-Agent": "hse-python-course"}
    async with aiohttp.ClientSession(timeout=timeout, headers=headers) as session:
        async with asyncio.TaskGroup() as group:
            tasks = [group.create_task(fetch_repo(session, name)) for name in names]
    return [task.result() for task in tasks]

repos = await fetch_repositories(["python/cpython", "psf/requests", "aio-libs/aiohttp"])
assert [repo["full_name"] for repo in repos] == ["python/cpython", "psf/requests", "aio-libs/aiohttp"]

## У ожидания должна быть граница

Тайм-аут может относиться к соединению, чтению одного ответа, отдельной операции или целому пакету. Это разные договоры. Клиентский `aiohttp.ClientTimeout` ограничивает HTTP-операции, а `asyncio.timeout()` удобно ограничивает произвольный async-блок.

> **Появилось в Python 3.11.** Контекстный менеджер `asyncio.timeout()` превращает внутреннюю отмену по истечении срока во внешний встроенный `TimeoutError`. Его ловят снаружи блока, если нужно вернуть предметную ошибку или выполнить fallback.

In [ ]:
async def bounded_report() -> str:
    try:
        async with asyncio.timeout(0.01):
            return await load_report("annual", 0.1)
    except TimeoutError:
        return "annual: timeout"

assert await bounded_report() == "annual: timeout"

## Отмена — запрос на остановку

`task.cancel()` не уничтожает задачу мгновенно. При следующей подходящей точке ожидания внутрь задачи поднимается `asyncio.CancelledError`. Coroutine должна освободить ресурс в `finally` и обычно снова пропустить отмену наружу.

`CancelledError` наследуется напрямую от `BaseException`, а не от `Exception`. Это защищает отмену от случайного `except Exception`. Намеренно перехватить и проглотить её можно, но это ломает ожидания `TaskGroup` и `asyncio.timeout()`; для обычного прикладного кода так делать не следует.

In [ ]:
cleanup_done = False

async def cancellable_work() -> None:
    global cleanup_done
    try:
        await asyncio.sleep(10)
    finally:
        cleanup_done = True

task = asyncio.create_task(cancellable_work())
await asyncio.sleep(0)
task.cancel()
try:
    await task
except asyncio.CancelledError:
    pass
assert cleanup_done and task.cancelled()

## Ошибка одной задачи и `ExceptionGroup`

Если несколько дочерних задач `TaskGroup` успели завершиться ошибками, группа поднимает `ExceptionGroup`. Конструкция `except* SomeError` обрабатывает подходящую подгруппу. И `ExceptionGroup`, и `except*` появились в Python 3.11 вместе со структурированной конкурентностью.

В прикладном сервисе часто разумнее не разбирать каждую ошибку на месте, а добавить контекст у границы одного запроса и позволить группе остановить связанную операцию целиком.

## Semaphore: конкурентность должна иметь предел

Если создать десять тысяч задач, это не означает, что удалённый API выдержит десять тысяч одновременных запросов. `asyncio.Semaphore(n)` пропускает внутрь критического блока не более `n` задач. Остальные не блокируют поток, а асинхронно ждут разрешения.

Semaphore используем через `async with`: разрешение вернётся даже при исключении или отмене. Лимит выбирают из ограничений API, соединений, памяти и измерений, а не по принципу «чем больше, тем быстрее».

In [ ]:
async def load_limited(name: str, semaphore: asyncio.Semaphore) -> str:
    async with semaphore:
        return await load_report(name, 0.01)

async def load_many(names: list[str], limit: int) -> list[str]:
    semaphore = asyncio.Semaphore(limit)
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(load_limited(name, semaphore)) for name in names]
    return [task.result() for task in tasks]

assert await load_many(["a", "b", "c", "d"], limit=2) == [
    "a: ready", "b: ready", "c: ready", "d: ready"
]

## Блокирующий код внутри async-функции

`time.sleep()`, `requests.get()` и обычное чтение большого файла не становятся неблокирующими только потому, что вызваны внутри `async def`. Они занимают поток event loop и замораживают соседние задачи.

Для библиотек выбирают нативный async-интерфейс. Короткую неизбежную блокирующую I/O-функцию можно вынести через `await asyncio.to_thread(func, ...)`. Для тяжёлого Python-вычисления поток обычно не решает задачу параллелизма обычной сборки CPython — нужны процессы или иной вычислительный backend.

## Как тестировать без сети

Предметная функция не обязана сама создавать HTTP-клиент. Передайте ей async-зависимость `fetch(url) -> dict`. В production это обёртка над `aiohttp`, а в тесте — маленькая coroutine с управляемым результатом, задержкой или исключением.

Так тест отдельно проверяет порядок результатов, тайм-аут, отмену соседей и лимит конкурентности. Публичный API остаётся интеграционной проверкой, а не условием прохождения каждого unit-теста.

## Неожиданно, но по правилам

### 1. Вызов `async def` ничего не запускает

Он создаёт coroutine-объект. Без `await` или task тело не выполнится, а при уничтожении объекта Python может предупредить, что coroutine никогда не ожидалась.

### 2. Два `await` подряд могут быть полностью последовательными

В выражении `[await first(), await second()]` второй вызов создаётся только после результата первого. Для перекрытия ожиданий задачи надо создать заранее или поместить в `TaskGroup`.

### 3. Порядок результата не обязан совпадать с порядком завершения

Мы возвращаем результаты по списку сохранённых task, поэтому быстрый третий запрос может закончиться первым, но остаться третьим в результате. Это удобный контракт, а не доказательство последовательности.

### 4. `404` — полученный HTTP-ответ

Сеть, DNS и сервер сработали достаточно хорошо, чтобы вернуть сообщение. Исключение появляется только после явного `raise_for_status()` или нашей проверки.

### 5. JSON может быть корректным и бесполезным

`{"stars": true}` синтаксически валиден. Более того, `isinstance(True, int)` истинно. Предметная валидация должна отдельно отклонить boolean там, где ожидается число звёзд.

### 6. Тайм-аут реализован через отмену

Если coroutine проглотит `CancelledError`, `asyncio.timeout()` и `TaskGroup` могут вести себя не так, как ожидает вызывающий код. Очистить ресурс — да; скрыть отмену — обычно нет.

### 7. Тысяча async-задач может быть хуже ста

Event loop способен их хранить, но API ответит `429`, пул соединений разрастётся, а память закончится. Semaphore — часть корректности, а не косметическая оптимизация.

### 8. `async` не ускоряет вычисление

Цикл без `await` не отдаёт управление. Добавлять бессмысленные `await asyncio.sleep(0)` внутрь вычисления — не замена подходящему исполнителю и алгоритму.

## Самопроверка

1. Чем HTTP отличается от JSON?
2. Какие части есть у запроса и ответа?
3. Почему `404` не является транспортной ошибкой?
4. Зачем явно задавать timeout?
5. Чем CPU-bound задача отличается от I/O-bound?
6. Чем конкурентность отличается от параллелизма?
7. Что возвращает вызов `async def`?
8. Когда `await` позволяет выполнить другую задачу?
9. Зачем нужен task?
10. Что гарантирует выход из `TaskGroup`?
11. Какие возможности раздела появились в Python 3.11?
12. Что изменилось у получения event loop в Python 3.14?
13. Почему одна `ClientSession` лучше сессии на каждый URL?
14. Как безопасно освобождать ресурс при отмене?
15. Почему `CancelledError` не следует проглатывать?
16. Что ограничивает semaphore?
17. Почему `requests.get` блокирует async-программу?
18. Как проверить async-логику без внешней сети?

## Источники

- [Обзор HTTP на MDN](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Overview) — устройство запросов и ответов.
- [Методы HTTP на MDN](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Methods) — семантика методов.
- [Документация `asyncio` Python 3.14](https://docs.python.org/3.14/library/asyncio.html) — высокоуровневые интерфейсы.
- [Coroutines and Tasks](https://docs.python.org/3.14/library/asyncio-task.html) — задачи, `TaskGroup`, отмена и timeout.
- [Synchronization Primitives](https://docs.python.org/3.14/library/asyncio-sync.html#asyncio.Semaphore) — semaphore.
- [Клиент `aiohttp`](https://docs.aiohttp.org/en/stable/client.html) — сессия, запрос и чтение ответа.

Версионные пометки относятся к стандартной библиотеке Python. `aiohttp` — отдельная зависимость курса.

## Итоги

- HTTP обменивается запросами и ответами; JSON является лишь одним из форматов тела.
- Тайм-аут и проверка статуса обязательны даже в маленьком клиенте.
- Асинхронность перекрывает независимые ожидания I/O, но не ускоряет CPU-bound код.
- Вызов coroutine не запускает её; task связывает coroutine с event loop.
- `TaskGroup` ограничивает время жизни связанных задач и появился в Python 3.11.
- Одна `aiohttp.ClientSession` повторно использует соединения.
- Отмена доставляется как `CancelledError`, а очистка выполняется в `finally`.
- `asyncio.timeout()` задаёт границу времени, semaphore — границу конкурентности.
- Тестируемая async-функция получает сетевую операцию как зависимость.

На семинаре эти правила применяются к настоящему GitHub API и к управляемым локальным сценариям ошибок.